# 04 - Loss Functions: Regression & Classification

## Learning Objectives
- Understand the role of loss functions in neural network training
- Implement and compare regression losses: MSE, MAE, Huber
- Implement and compare classification losses: BCE, cross-entropy
- Know when to use each loss function

## Prerequisites
- Notebooks 01-03 (forward/backward propagation)
- Basic calculus (derivatives)

## Table of Contents
1. [Why Loss Functions Matter](#1)
2. [Regression Losses](#2)
3. [Classification Losses](#3)
4. [PyTorch Loss Functions](#4)
5. [Choosing the Right Loss](#5)
6. [Exercises](#6)
7. [Common Mistakes & Debugging Tips](#7)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

np.random.seed(42)
torch.manual_seed(42)

<a id='1'></a>
## 1. Why Loss Functions Matter

A **loss function** (cost function, objective function) measures how far our model's predictions are from the true values.

- Training = minimizing the loss function via gradient descent
- Different tasks need different losses
- The loss function shapes the gradient landscape and affects convergence

<a id='2'></a>
## 2. Regression Losses

### 2.1 Mean Squared Error (MSE)

$$L_{\text{MSE}} = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$$

- Penalizes large errors quadratically → sensitive to outliers
- Gradient: $\frac{\partial L}{\partial \hat{y}_i} = -\frac{2}{n}(y_i - \hat{y}_i)$
- Most common regression loss

### 2.2 Mean Absolute Error (MAE)

$$L_{\text{MAE}} = \frac{1}{n}\sum_{i=1}^{n}|y_i - \hat{y}_i|$$

- Linear penalty → robust to outliers
- Gradient is constant magnitude (±1) → can be unstable near zero

### 2.3 Huber Loss (Smooth L1)

$$L_{\delta}(a) = \begin{cases} \frac{1}{2}a^2 & \text{if } |a| \leq \delta \\ \delta(|a| - \frac{1}{2}\delta) & \text{otherwise} \end{cases}$$

- Combines MSE (small errors) and MAE (large errors)
- Best of both worlds: smooth gradients near zero, robust to outliers

In [ ]:
# Implement regression losses from scratch
def mse_loss(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def mae_loss(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def huber_loss(y_true, y_pred, delta=1.0):
    error = y_true - y_pred
    is_small = np.abs(error) <= delta
    squared = 0.5 * error ** 2
    linear = delta * (np.abs(error) - 0.5 * delta)
    return np.mean(np.where(is_small, squared, linear))

# Visualize how each loss responds to different error magnitudes
errors = np.linspace(-4, 4, 200)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (name, fn) in zip(axes, [('MSE', lambda e: e**2), 
                                    ('MAE', lambda e: np.abs(e)),
                                    ('Huber (δ=1)', lambda e: np.where(np.abs(e)<=1, 0.5*e**2, np.abs(e)-0.5))]):
    ax.plot(errors, fn(errors), linewidth=2)
    ax.set_title(name, fontsize=13)
    ax.set_xlabel('Error (y - ŷ)')
    ax.set_ylabel('Loss')
    ax.grid(True, alpha=0.3)
    ax.axvline(0, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Demonstrate outlier sensitivity
y_true = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
y_pred = np.array([1.1, 2.2, 2.8, 4.1, 4.9])  # good predictions

print("=== Without outlier ===")
print(f"MSE:   {mse_loss(y_true, y_pred):.4f}")
print(f"MAE:   {mae_loss(y_true, y_pred):.4f}")
print(f"Huber: {huber_loss(y_true, y_pred):.4f}")

# Add an outlier
y_true_out = np.array([1.0, 2.0, 3.0, 4.0, 50.0])  # outlier!
y_pred_out = np.array([1.1, 2.2, 2.8, 4.1, 4.9])

print("\n=== With outlier (y=50, ŷ=4.9) ===")
print(f"MSE:   {mse_loss(y_true_out, y_pred_out):.4f}  ← huge increase")
print(f"MAE:   {mae_loss(y_true_out, y_pred_out):.4f}")
print(f"Huber: {huber_loss(y_true_out, y_pred_out):.4f}")

<a id='3'></a>
## 3. Classification Losses

### 3.1 Binary Cross-Entropy (BCE)

For binary classification where $\hat{y} \in (0,1)$ (after sigmoid):

$$L_{\text{BCE}} = -\frac{1}{n}\sum_{i=1}^{n}\left[y_i\log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i)\right]$$

### 3.2 BCEWithLogitsLoss (numerically stable)

Combines sigmoid + BCE in one step. Uses the log-sum-exp trick for numerical stability.

**Always prefer BCEWithLogitsLoss over manual sigmoid + BCE.**

### 3.3 Categorical Cross-Entropy (multi-class)

$$L_{\text{CE}} = -\sum_{c=1}^{C} y_c \log(\hat{y}_c)$$

In PyTorch: `nn.CrossEntropyLoss` = softmax + NLLLoss (takes raw logits).

### 3.4 Label Smoothing

Instead of hard labels (0 or 1), use soft labels: $y_{\text{smooth}} = (1-\epsilon)y + \epsilon/C$

- Prevents overconfident predictions
- Acts as regularization
- PyTorch: `nn.CrossEntropyLoss(label_smoothing=0.1)`

In [ ]:
# BCE from scratch
def bce_loss(y_true, y_pred, eps=1e-7):
    y_pred = np.clip(y_pred, eps, 1 - eps)  # numerical stability
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

# Cross-entropy from scratch
def cross_entropy_loss(y_true_indices, logits, eps=1e-7):
    """y_true_indices: class indices, logits: raw scores (before softmax)"""
    exp_logits = np.exp(logits - np.max(logits, axis=1, keepdims=True))  # stable softmax
    probs = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)
    n = len(y_true_indices)
    return -np.mean(np.log(probs[np.arange(n), y_true_indices] + eps))

# Example: binary classification
y_true_bin = np.array([1, 0, 1, 1, 0])
y_pred_bin = np.array([0.9, 0.1, 0.8, 0.7, 0.3])
print(f"BCE Loss: {bce_loss(y_true_bin, y_pred_bin):.4f}")

# Example: multi-class
y_true_mc = np.array([0, 2, 1])  # class indices
logits = np.array([[2.0, 0.5, 0.1],
                   [0.1, 0.3, 2.5],
                   [0.5, 2.0, 0.3]])
print(f"CE Loss:  {cross_entropy_loss(y_true_mc, logits):.4f}")

<a id='4'></a>
## 4. PyTorch Loss Functions

In [ ]:
# PyTorch equivalents
y = torch.tensor([1.0, 2.0, 3.0, 4.0])
y_hat = torch.tensor([1.1, 2.3, 2.7, 4.2])

print("=== Regression ===")
print(f"MSE:   {nn.MSELoss()(y_hat, y).item():.4f}")
print(f"MAE:   {nn.L1Loss()(y_hat, y).item():.4f}")
print(f"Huber: {nn.SmoothL1Loss()(y_hat, y).item():.4f}")

print("\n=== Classification ===")
# Binary
logits_bin = torch.tensor([2.0, -1.0, 1.5])
targets_bin = torch.tensor([1.0, 0.0, 1.0])
print(f"BCEWithLogits: {nn.BCEWithLogitsLoss()(logits_bin, targets_bin).item():.4f}")

# Multi-class (CrossEntropyLoss takes raw logits + class indices)
logits_mc = torch.tensor([[2.0, 0.5, 0.1], [0.1, 0.3, 2.5]])
targets_mc = torch.tensor([0, 2])
print(f"CrossEntropy:  {nn.CrossEntropyLoss()(logits_mc, targets_mc).item():.4f}")

# With label smoothing
print(f"CE + smoothing: {nn.CrossEntropyLoss(label_smoothing=0.1)(logits_mc, targets_mc).item():.4f}")

<a id='5'></a>
## 5. Choosing the Right Loss

| Task | Loss Function | Input to Loss |
|------|--------------|---------------|
| Regression | `nn.MSELoss()` | predictions, targets (floats) |
| Regression with outliers | `nn.SmoothL1Loss()` | predictions, targets (floats) |
| Binary classification | `nn.BCEWithLogitsLoss()` | raw logits, targets (0/1 floats) |
| Multi-class classification | `nn.CrossEntropyLoss()` | raw logits, class indices (longs) |
| Multi-label classification | `nn.BCEWithLogitsLoss()` | raw logits, multi-hot targets |

<a id='6'></a>
## 6. Exercises

**Exercise 1:** Implement Huber loss from scratch and verify it matches `nn.SmoothL1Loss`.

**Exercise 2:** Plot BCE loss as a function of predicted probability for y=1 and y=0.

**Exercise 3:** Show that `nn.CrossEntropyLoss` = `nn.LogSoftmax` + `nn.NLLLoss`.

In [ ]:
# Solution 1: Huber loss verification
def huber_torch(y_true, y_pred, delta=1.0):
    error = y_true - y_pred
    abs_error = torch.abs(error)
    quadratic = 0.5 * error ** 2
    linear = delta * (abs_error - 0.5 * delta)
    return torch.mean(torch.where(abs_error <= delta, quadratic, linear))

y = torch.tensor([1.0, 2.0, 3.0])
yh = torch.tensor([1.5, 1.0, 3.5])
print(f"Custom Huber:  {huber_torch(y, yh):.4f}")
print(f"SmoothL1Loss:  {nn.SmoothL1Loss()(yh, y):.4f}")

In [ ]:
# Solution 2: BCE loss plot
p = np.linspace(0.01, 0.99, 100)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(p, -np.log(p), label='y=1: -log(p)', linewidth=2)
ax.plot(p, -np.log(1-p), label='y=0: -log(1-p)', linewidth=2)
ax.set_xlabel('Predicted probability')
ax.set_ylabel('Loss')
ax.set_title('Binary Cross-Entropy')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

<a id='7'></a>
## 7. Common Mistakes & Debugging Tips

| Mistake | Fix |
|---------|-----|
| Using sigmoid + `nn.BCELoss` | Use `nn.BCEWithLogitsLoss` (numerically stable) |
| Passing probabilities to `nn.CrossEntropyLoss` | Pass raw logits (it applies softmax internally) |
| Wrong target dtype for cross-entropy | Targets must be `torch.long` (class indices) |
| Wrong target dtype for BCE | Targets must be `torch.float` |
| NaN loss | Check for log(0): clip predictions or use LogSoftmax |
| Loss not decreasing | Check learning rate, loss function choice, data correctness |